In [1]:
from pathlib import Path
# === Imports ===
from pathlib import Path
import torch
import torch.backends.cudnn as cudnn
import time

import sys
import os
# Add the project root to sys.path to allow imports from other folders (e.g., 'runner', 'data_loader', etc.)
sys.path.append(os.path.abspath('..'))  # Assumes this notebook is in /tutorials/

from runner.experiment import Experiment  # Main class to manage training and testing



In [2]:
# --- Stage manifest -> pair_<d0>/ folders WGAST's loader expects -------------
#
# Set CITY and re-run. Raw sources expected at:
#   ./data/secondary/raw/{MODIS_d0,MODIS_t0,Landsat8_t0,Sentinel2_t0}_<Cap>/
# Manifest of (t0, d0) pairs at ./data/secondary/manifest_<Cap>.parquet.
# The S2 reference scene comes from manifest["t0_s2"] (±2-day bundle, 01s) but
# is staged under the t0 filename so the pair stays internally consistent.
#
# RESUMABLE: d0 days whose wgast_<d0>.tif is already in the cache are skipped,
# so a year-long city can be processed in several sittings.

CITY = "Rome"   

import shutil
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import Affine
from rasterio.crs import CRS as RioCRS
from pathlib import Path

# MODIS Sinusoidal on the authalic SPHERE (R=6371007.181 m). GEE-exported MODIS
# GeoTIFFs are often mislabelled with the WGS84 ellipsoid, which places the data
# ~18 km north of the AOI; reprojecting from the wrong CRS then yields all-zeros.
# warp() below overrides any Sinusoidal source CRS with this correct one.
MODIS_SINU = RioCRS.from_proj4("+proj=sinu +lon_0=0 +x_0=0 +y_0=0 "
                              "+R=6371007.181 +units=m +no_defs")

CAP = CITY.capitalize()
SRC = Path("./data/secondary/raw")
DST = Path(f"./data/wgast/Tdivision/test_{CAP}")
CACHE = Path(f"./data/secondary/wgast_cache/{CITY.lower()}")
DST.mkdir(parents=True, exist_ok=True)
for d in DST.glob("pair_*"):
    shutil.rmtree(d)

manifest = pd.read_parquet(f"./data/secondary/manifest_{CAP}.parquet")
if "t0_s2" not in manifest.columns:   # manifest from the strict same-day era
    manifest["t0_s2"] = manifest["t0"]

# Anchor grid: first Landsat scene's CRS+origin. 30m, 400x400.
ref = next((SRC / f"Landsat8_t0_{CAP}").glob("*.tif"))
with rasterio.open(ref) as r:
    CRS, T30 = r.crs, r.transform
H30, W30 = 400, 400
T10 = Affine(T30.a / 3, 0, T30.c, 0, T30.e / 3, T30.f)
H10, W10 = H30 * 3, W30 * 3

def warp(src_path, dst_path, T, H, W, bands):
    with rasterio.open(src_path) as s:
        # NB: reproject() reads georeferencing from a rasterio.band's dataset and
        # IGNORES the src_crs kwarg, so we read into arrays and pass src_crs
        # explicitly -- otherwise the Sinusoidal override below is a silent no-op.
        src_crs = s.crs
        if src_crs is not None and "Sinusoidal" in (src_crs.to_wkt() or ""):
            src_crs = MODIS_SINU      # fix mislabelled MODIS (WGS84-ellipsoid tag)
        src_transform = s.transform
        src_arr = s.read().astype(np.float32)      # (bands, h, w)
    arr = np.zeros((bands, H, W), dtype=np.float32)
    for b in range(bands):
        reproject(
            source=src_arr[b], destination=arr[b],
            src_transform=src_transform, src_crs=src_crs,
            dst_transform=T, dst_crs=CRS,
            resampling=Resampling.bilinear,
        )
    # Sanity guard: a fully-empty warp means the source didn't overlap the AOI
    # (the exact bug this cell fixes). Surface it loudly instead of silently
    # writing zeros that collapse every WGAST output to the t0 reference.
    if not np.any(arr):
        print(f"  [WARN] {Path(dst_path).name}: warped raster is ALL ZERO "
              f"(source/AOI overlap problem?) src={Path(src_path).name}")
    with rasterio.open(dst_path, "w", driver="GTiff",
                       height=H, width=W, count=bands, dtype="float32",
                       crs=CRS, transform=T) as d:
        d.write(arr)

modis_t0_by_date = {p.stem.replace("_", ""): p for p in (SRC / f"MODIS_t0_{CAP}").glob("*.tif")}
modis_d0_by_date = {p.stem.replace("_", ""): p for p in (SRC / f"MODIS_d0_{CAP}").glob("*.tif")}
landsat_by_date  = {p.stem[-8:]:             p for p in (SRC / f"Landsat8_t0_{CAP}").glob("*.tif")}
s2_by_date       = {p.name[:8]:              p for p in (SRC / f"Sentinel2_t0_{CAP}").glob("*.tif")}

n_ok = n_skip = n_cached = 0
for _, row in manifest.iterrows():
    d0, t0, t0_s2 = row["d0"], row["t0"], row["t0_s2"]
    fl_t0, fl_d0 = t0.strftime("%Y%m%d"), d0.strftime("%Y%m%d")
    fl_s2 = t0_s2.strftime("%Y%m%d")
    if (CACHE / f"wgast_{fl_d0}.tif").exists():
        n_cached += 1
        continue
    srcs = {
        "00_MODIS":    modis_t0_by_date.get(fl_t0),
        "00_Landsat":  landsat_by_date.get(fl_t0),
        "00_Sentinel": s2_by_date.get(fl_s2),     # ±2-day bundle scene
        "01_MODIS":    modis_d0_by_date.get(fl_d0),
    }
    if not all(p and p.exists() for p in srcs.values()):
        n_skip += 1
        continue

    out = DST / f"pair_{fl_d0}"
    out.mkdir(exist_ok=True)

    warp(srcs["00_MODIS"],    out / f"00_MODIS_{fl_t0}.tif",    T10, H10, W10, 1)
    warp(srcs["00_Sentinel"], out / f"00_Sentinel_{fl_t0}.tif", T10, H10, W10, 3)
    warp(srcs["01_MODIS"],    out / f"01_MODIS_{fl_d0}.tif",    T10, H10, W10, 1)
    warp(srcs["00_Landsat"],  out / f"00_Landsat_{fl_t0}.tif",  T30, H30, W30, 4)

    # 01_Landsat@d0 stub (unused as input; only its filename is consumed)
    with rasterio.open(out / f"01_Landsat_{fl_d0}.tif", "w", driver="GTiff",
                       height=H30, width=W30, count=1, dtype="float32",
                       crs=CRS, transform=T30) as d:
        d.write(np.zeros((1, H30, W30), dtype=np.float32))

    # all-ones masks at each tif's native shape
    for f in out.glob("*.tif"):
        for sat in ("MODIS", "Landsat", "Sentinel"):
            if f"_{sat}_" in f.stem:
                with rasterio.open(f) as s:
                    shp = (s.height, s.width)
                mname = f.stem.replace(f"_{sat}_", f"_{sat}_mask_") + ".npy"
                np.save(f.parent / mname, np.ones(shp, dtype=np.float32))
                break
    n_ok += 1

print(f"[{CITY}] staged {n_ok} pairs into {DST}  "
      f"(skipped {n_skip} rows missing an input, {n_cached} already cached)")

[Rome] staged 28 pairs into data/wgast/Tdivision/test_Rome  (skipped 0 rows missing an input, 0 already cached)


In [3]:
from pathlib import Path
import shutil
import sys, os
import torch
sys.path.append(os.path.abspath(".."))
from runner.experiment import Experiment

CAP = CITY.capitalize()   # CITY is set in the staging cell above

class Options:
    lr=2e-4; batch_size=1; epochs=0
    cuda=torch.cuda.is_available(); ngpu=1; num_workers=0
    save_dir   = Path("data/wgast/Tdivision")
    image_size = [400, 400]
    ifAdaIN=True; ifAttention=True; ifTwoInput=False
    a=1e-2; b=1; c=1; d=1

test_dir  = Path(f"data/wgast/Tdivision/test_{CAP}")
cache_dir = Path(f"data/secondary/wgast_cache/{CITY.lower()}")
cache_dir.mkdir(parents=True, exist_ok=True)

experiment = Experiment(Options())
experiment.test(test_dir, patch_size=[32,32], num_workers=0)

for p in test_dir.glob("01_Sentinel_*.tif"):
    d0 = p.stem.split("_")[-1]
    shutil.move(str(p), cache_dir / f"wgast_{d0}.tif")

print(f"[{CITY}] WGAST outputs cached -> {cache_dir}")


Model initialization
There are 7884208 trainable parameters for generator.
There are 2765505 trainable parameters for nlayerdiscriminator.
*****************
Testing...
Start test for image :  01_Sentinel_20220925.tif
Time cost: 0.13346970902057365s
End test for image :  01_Sentinel_20220925.tif
*****************************************************
Start test for image :  01_Sentinel_20220913.tif
Time cost: 0.2271031669806689s
End test for image :  01_Sentinel_20220913.tif
*****************************************************
Start test for image :  01_Sentinel_20220912.tif
Time cost: 0.17661141697317362s
End test for image :  01_Sentinel_20220912.tif
*****************************************************
Start test for image :  01_Sentinel_20220923.tif
Time cost: 0.17215800000121817s
End test for image :  01_Sentinel_20220923.tif
*****************************************************
Start test for image :  01_Sentinel_20220906.tif
Time cost: 0.1423178329714574s
End test for image :  01_